# Graph RAG – Restaurant Recommendation (improved)

Notebook này nâng bản prototype trước thành một pipeline **Graph RAG hoàn chỉnh hơn**:

- đọc dữ liệu thật từ 3 file CSV
- chuẩn hóa schema và hợp nhất nguồn dữ liệu
- trích xuất **aspect-based sentiment**
- xây **Knowledge Graph sâu hơn** với node riêng cho `Cuisine`, `Category`, `PriceBand`, `AtmosphereTag`
- tạo **2 tầng embedding**
  - `restaurant_summary` embeddings
  - `review_chunk` embeddings
- hỗ trợ **hybrid retrieval**
  - graph filtering
  - vector retrieval
  - neighborhood / subgraph expansion
  - fusion reranking
- thêm **evaluation cho retrieval quality**
  - Recall@K
  - MRR@K
  - nDCG@K

> Embedding **có** và vẫn là phần trung tâm cho semantic retrieval; graph dùng để tăng tính cấu trúc, explainability và neighborhood expansion.


## 0. Cài thư viện

In [ ]:
%pip install -q \
    pandas==2.2.2 numpy==1.26.4 tqdm==4.66.4 pydantic==2.7.4 python-dotenv==1.0.1 \
    neo4j==5.19.0 qdrant-client==1.9.1 sentence-transformers==3.0.1 torch==2.3.1 \
    langchain==0.2.6 langchain-community==0.2.6 langchain-openai==0.1.13 langchain-anthropic==0.1.20
print("✅ Dependencies installed")


## 1. Imports, config và đường dẫn

In [1]:
from __future__ import annotations
import os, re, json, ast, math, hashlib, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()

# ------------------------------
# Repo-aware paths (Windows-friendly)
# ------------------------------
# README.md (repo root) documents these input files:
# - be_google_maps_unique.csv
# - store_feedback_crawled.csv
# - foody_hust_output/foody_hust_places_from_store_csv.csv
# Some earlier notebook versions used /mnt/data/* (Kaggle/Colab style). Here we default to local repo paths.

REPO_ROOT = Path.cwd()
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(REPO_ROOT)))

GOOGLE_MAPS_PATH = Path(os.getenv("GOOGLE_MAPS_PATH", str(DATA_ROOT / "be_google_maps_unique.csv")))
FEEDBACK_PATH = Path(os.getenv("FEEDBACK_PATH", str(DATA_ROOT / "store_feedback_crawled.csv")))
FOODY_PATH = Path(os.getenv(
    "FOODY_PATH",
    str(DATA_ROOT / "foody_hust_output" / "foody_hust_places_from_store_csv.csv"),
))

# ------------------------------
# Neo4j config (match README)
# ------------------------------
# README uses docker-compose.neo4j.yml default bolt:7687 and browser:7474.
# Keep both NEO4J_USER and NEO4J_USERNAME variants for compatibility.

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", os.getenv("NEO4J_USERNAME", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j123")

# ------------------------------
# Vector DB config (Qdrant) - optional
# ------------------------------
QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
COLL_RESTAURANT = os.getenv("COLL_RESTAURANT", "restaurant_summary_v2")
COLL_REVIEW = os.getenv("COLL_REVIEW", "review_chunks_v2")

# ------------------------------
# Embedding/LLM config
# ------------------------------
EMBED_MODEL = os.getenv("EMBED_MODEL", "intfloat/multilingual-e5-base")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

print("✅ Repo paths configured")
print("   Repo root:", REPO_ROOT)
print("   Data root:", DATA_ROOT)
print("   GMaps   :", GOOGLE_MAPS_PATH)
print("   Feedback:", FEEDBACK_PATH)
print("   Foody   :", FOODY_PATH)
print("✅ Neo4j:", NEO4J_URI, "user=", NEO4J_USER)
print("✅ Embed model:", EMBED_MODEL)

# Basic existence checks (helpful on Windows)
for p in [GOOGLE_MAPS_PATH, FEEDBACK_PATH, FOODY_PATH]:
    if not p.exists():
        print("⚠️ Missing file:", p)


c:\Users\Admin\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Admin\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


✅ Paths configured
   GMaps   : /mnt/data/be_google_maps_unique.csv
   Feedback: /mnt/data/store_feedback_crawled.csv
   Foody   : /mnt/data/foody_hust_places_from_store_csv.csv
✅ Embed model: intfloat/multilingual-e5-base


## 2. Inspect dữ liệu thật

In [ ]:
raw_gmaps = pd.read_csv(GOOGLE_MAPS_PATH)
raw_feedback = pd.read_csv(FEEDBACK_PATH)
raw_foody = pd.read_csv(FOODY_PATH)

display(raw_gmaps.head(2))
display(raw_feedback.head(2))
display(raw_foody.head(2))

print("Shapes:")
print("  raw_gmaps   :", raw_gmaps.shape)
print("  raw_feedback:", raw_feedback.shape)
print("  raw_foody   :", raw_foody.shape)


In [ ]:
def _is_nan(x: Any) -> bool:
    return pd.isna(x)

def normalize_text(x: Any) -> str:
    if x is None or _is_nan(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKC", x)
    return re.sub(r"\s+", " ", x)

def slugify_vn(x: Any) -> str:
    x = normalize_text(x)
    x = ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn')
    x = x.replace("đ", "d")
    x = re.sub(r"[^a-z0-9]+", "-", x).strip("-")
    return x

def to_float(x: Any) -> Optional[float]:
    if x is None or _is_nan(x) or str(x).strip() == "":
        return None
    try:
        return float(x)
    except:
        s = str(x).replace(".", "").replace(",", ".")
        s = re.sub(r"[^0-9.\-]", "", s)
        try:
            return float(s)
        except:
            return None

def to_int(x: Any) -> Optional[int]:
    v = to_float(x)
    return None if v is None else int(v)

def parse_jsonish(x: Any) -> Any:
    if x is None or _is_nan(x):
        return None
    if isinstance(x, (list, dict)):
        return x
    s = str(x).strip()
    if not s:
        return None
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except:
            pass
    return s

def split_semi(x: Any) -> List[str]:
    if x is None or _is_nan(x):
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    parsed = parse_jsonish(x)
    if isinstance(parsed, list):
        return [str(i).strip() for i in parsed if str(i).strip()]
    s = str(x)
    parts = re.split(r"[;|,/]", s)
    return [p.strip() for p in parts if p and p.strip()]

def parse_price_band(text: Any) -> Optional[str]:
    if text is None or _is_nan(text):
        return None
    s = normalize_text(text)
    if not s:
        return None
    nums = re.findall(r"\d[\d\.]*", s)
    vals = []
    for n in nums:
        try:
            vals.append(int(float(n.replace(".", ""))))
        except:
            pass
    if not vals:
        return None
    hi = max(vals)
    if hi <= 50000:
        return "budget"
    if hi <= 120000:
        return "mid"
    return "premium"


## 3. Canonicalize schema và hợp nhất nguồn dữ liệu

In [ ]:
def canonicalize_gmaps(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["store_id"] = df["id"].astype("Int64").astype(str)
    out["store_name"] = df["name"].fillna(df.get("query_name"))
    out["query_name"] = df.get("query_name")
    out["address"] = df["address"].fillna(df.get("query_address"))
    out["query_address"] = df.get("query_address")
    out["lat"] = df["latitude"].apply(to_float)
    out["lng"] = df["longitude"].apply(to_float)
    out["gmaps_rating"] = df["rating"].apply(to_float)
    out["gmaps_review_count"] = df["review_count"].apply(to_int)
    out["price_text"] = df.get("price")
    out["price_band"] = df.get("price", pd.Series([None]*len(df))).apply(parse_price_band)
    out["phone"] = df.get("phone")
    out["website"] = df.get("website")
    out["type_text"] = df.get("type")
    out["categories"] = df.get("type", pd.Series([None]*len(df))).apply(split_semi)
    out["opening_hours_raw"] = df.get("opening_hours")
    out["service_options_raw"] = df.get("service_options")
    out["atmosphere"] = df.get("atmosphere", pd.Series([None]*len(df))).apply(split_semi)
    out["crowd"] = df.get("crowd", pd.Series([None]*len(df))).apply(split_semi)
    out["popular_for"] = df.get("popular_for", pd.Series([None]*len(df))).apply(split_semi)
    out["dining_options"] = df.get("dining_options", pd.Series([None]*len(df))).apply(split_semi)
    out["offerings"] = df.get("offerings", pd.Series([None]*len(df))).apply(split_semi)
    out["store_key"] = out["store_id"].astype(str)
    out["name_norm"] = out["store_name"].apply(slugify_vn)
    out["addr_norm"] = out["address"].apply(slugify_vn)
    return out

def canonicalize_feedback(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "crawl_status" in df.columns:
        df = df[df["crawl_status"].fillna("ok").eq("ok")].copy()
    out = pd.DataFrame()
    out["store_id"] = df["store_id"].astype("Int64").astype(str)
    out["store_name"] = df["store_name"]
    out["rated_at"] = df.get("rated_at")
    out["rating"] = df["rating"].apply(to_float).fillna(3.0)
    out["feedback"] = df["feedback"].fillna("").astype(str)
    out["source"] = df.get("source", pd.Series(["delivery"]*len(df))).fillna("delivery")
    out["review_id"] = [
        hashlib.md5(f"{sid}|{rt}|{fb}".encode("utf-8")).hexdigest()[:16]
        for sid, rt, fb in zip(out["store_id"], out["rated_at"], out["feedback"])
    ]
    out["store_key"] = out["store_id"].astype(str)
    out["name_norm"] = out["store_name"].apply(slugify_vn)
    return out

def canonicalize_foody(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["input_store_id"] = df["input_store_id"].astype("Int64").astype(str)
    out["input_store_name"] = df["input_store_name"]
    out["crawl_status"] = df.get("crawl_status")
    out["foody_name"] = df.get("name")
    out["foody_address"] = df.get("address")
    out["district"] = df.get("district")
    out["area"] = df.get("area")
    out["city"] = df.get("city")
    out["foody_lat"] = df.get("lat").apply(to_float)
    out["foody_lng"] = df.get("lng").apply(to_float)
    out["foody_rating"] = df.get("avg_rating").apply(to_float)
    out["foody_review_count"] = df.get("total_review").apply(to_int)
    out["price_min"] = df.get("price_min").apply(to_float)
    out["price_max"] = df.get("price_max").apply(to_float)
    out["foody_price_band"] = df.get("price_max").apply(
        lambda x: "budget" if (to_float(x) or 0) <= 50000 else ("mid" if (to_float(x) or 0) <= 120000 else "premium")
        if to_float(x) is not None else None
    )
    out["categories"] = df.get("categories", pd.Series([None]*len(df))).apply(split_semi)
    out["cuisines"] = df.get("cuisines", pd.Series([None]*len(df))).apply(split_semi)
    out["audiences"] = df.get("audiences", pd.Series([None]*len(df))).apply(split_semi)
    out["opening_hours_foody"] = df.get("opening_hours")
    out["rating_quality"] = df.get("rating_quality").apply(to_float)
    out["rating_position"] = df.get("rating_position").apply(to_float)
    out["rating_service"] = df.get("rating_service").apply(to_float)
    out["rating_price"] = df.get("rating_price").apply(to_float)
    out["rating_space"] = df.get("rating_space").apply(to_float)
    out["store_key"] = out["input_store_id"].astype(str)
    out["name_norm"] = out["input_store_name"].apply(slugify_vn)
    return out

gmaps = canonicalize_gmaps(raw_gmaps)
feedback = canonicalize_feedback(raw_feedback)
foody = canonicalize_foody(raw_foody)


In [ ]:
restaurants = gmaps.merge(
    foody.drop_duplicates("store_key"),
    on="store_key",
    how="left",
    suffixes=("", "_foody")
)

restaurants["name"] = restaurants["store_name"].fillna(restaurants["foody_name"]).fillna(restaurants["query_name"])
restaurants["address_final"] = restaurants["address"].fillna(restaurants["foody_address"]).fillna(restaurants["query_address"])
restaurants["district_final"] = restaurants["district"]
restaurants["city_final"] = restaurants["city"].fillna("Hà Nội")
restaurants["price_band_final"] = restaurants["price_band"].fillna(restaurants["foody_price_band"])

def merge_list_cols(a, b):
    aa = a if isinstance(a, list) else []
    bb = b if isinstance(b, list) else []
    out, seen = [], set()
    for x in aa + bb:
        x = str(x).strip()
        if x and x.lower() not in seen:
            out.append(x)
            seen.add(x.lower())
    return out

restaurants["categories_final"] = [merge_list_cols(a, b) for a, b in zip(restaurants["categories"], restaurants["categories_foody"])]
restaurants["cuisines_final"] = restaurants["cuisines"].apply(lambda x: x if isinstance(x, list) else [])
restaurants["audiences_final"] = [merge_list_cols(a, b) for a, b in zip(restaurants["crowd"], restaurants["audiences"])]

summary = pd.DataFrame({
    "store_key": restaurants["store_key"],
    "name": restaurants["name"],
    "address": restaurants["address_final"],
    "district": restaurants["district_final"],
    "city": restaurants["city_final"],
    "gmaps_rating": restaurants["gmaps_rating"],
    "foody_rating": restaurants["foody_rating"],
    "price_band": restaurants["price_band_final"],
    "categories": restaurants["categories_final"],
    "cuisines": restaurants["cuisines_final"],
    "atmosphere": restaurants["atmosphere"],
    "audiences": restaurants["audiences_final"],
    "lat": restaurants["lat"],
    "lng": restaurants["lng"],
})
display(summary.head(10))
print("✅ Unified restaurant table ready:", summary.shape)


## 4. Aspect-based sentiment và review evidence

In [ ]:
ASPECT_KEYWORDS = {
    "food_quality": {"pos": ["ngon", "rất ngon", "vừa miệng", "đậm đà", "thơm", "chất lượng", "đáng tiền", "khá ngon"],
                     "neg": ["dở", "nhạt", "mặn", "nguội", "ôi", "thiu", "không ngon", "sai", "thiếu món", "không hợp khẩu vị"]},
    "service": {"pos": ["nhanh", "thân thiện", "nhiệt tình", "chu đáo", "lễ phép", "phục vụ tốt"],
                "neg": ["chậm", "lâu", "thái độ", "khó chịu", "cộc", "quát", "không nhiệt tình", "phục vụ chậm"]},
    "cleanliness": {"pos": ["sạch", "sạch sẽ", "thoáng", "gọn", "mát"],
                    "neg": ["bẩn", "hôi", "ruồi", "dơ", "ám mùi"]},
    "packaging": {"pos": ["đóng gói tốt", "cẩn thận", "gói kỹ", "không đổ"],
                  "neg": ["đổ", "rò", "rách", "thiếu", "vỡ", "bung"]},
    "price": {"pos": ["rẻ", "hợp lý", "đáng tiền", "vừa túi tiền"],
              "neg": ["đắt", "chát", "mắc", "không đáng"]},
    "space": {"pos": ["đẹp", "rộng", "ấm cúng", "yên tĩnh", "thoải mái"],
              "neg": ["chật", "ồn", "nóng", "bí", "đông quá"]},
    "speed": {"pos": ["nhanh", "lẹ", "ra món nhanh"],
              "neg": ["lâu", "chờ lâu", "ra món chậm", "ship lâu"]},
}

def normalize_review_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s.lower())
    s = re.sub(r"\s+", " ", s).strip()
    return s

def detect_aspects(text: str) -> Dict[str, float]:
    txt = normalize_review_text(text)
    scores = {}
    for aspect, kws in ASPECT_KEYWORDS.items():
        pos = sum(1 for w in kws["pos"] if w in txt)
        neg = sum(1 for w in kws["neg"] if w in txt)
        if pos + neg > 0:
            scores[aspect] = round((pos - neg) / (pos + neg), 3)
    return scores

def classify_sentiment(rating: float, aspect_scores: Dict[str, float]) -> str:
    if aspect_scores:
        avg = sum(aspect_scores.values()) / len(aspect_scores)
        if avg > 0.15:
            return "positive"
        if avg < -0.15:
            return "negative"
    if rating >= 4:
        return "positive"
    if rating >= 3:
        return "neutral"
    return "negative"

feedback_proc = feedback.copy()
feedback_proc["aspect_scores"] = feedback_proc["feedback"].apply(detect_aspects)
feedback_proc["sentiment"] = [classify_sentiment(r, a) for r, a in zip(feedback_proc["rating"], feedback_proc["aspect_scores"])]
feedback_proc["feedback_norm"] = feedback_proc["feedback"].apply(normalize_review_text)


In [ ]:
attr_rows = []
for store_key, grp in feedback_proc.groupby("store_key"):
    bucket = defaultdict(list)
    for asp_map in grp["aspect_scores"]:
        for k, v in asp_map.items():
            bucket[k].append(v)
    for aspect, vals in bucket.items():
        attr_rows.append({
            "store_key": store_key,
            "attribute_type": aspect,
            "attribute_score": round(float(np.mean(vals)), 3),
            "sample_count": len(vals)
        })
restaurant_attrs = pd.DataFrame(attr_rows)
display(restaurant_attrs.head(10))


In [ ]:
def build_review_chunk_text(row: pd.Series, restaurant_name: str = "") -> str:
    aspects = ", ".join(f"{k}={v:+.2f}" for k, v in (row["aspect_scores"] or {}).items())
    return (
        f"Quan: {restaurant_name or row['store_name']}\n"
        f"Rating: {row['rating']}\n"
        f"Sentiment: {row['sentiment']}\n"
        f"Aspects: {aspects}\n"
        f"Review: {row['feedback']}"
    )

name_map = summary.set_index("store_key")["name"].to_dict()
review_chunks = feedback_proc.copy()
review_chunks["chunk_id"] = review_chunks["review_id"]
review_chunks["chunk_text"] = [build_review_chunk_text(r, name_map.get(r["store_key"], r["store_name"])) for _, r in review_chunks.iterrows()]
display(review_chunks[["store_key","chunk_id","chunk_text"]].head(3))


## 5. Graph schema sâu hơn

In [ ]:
GRAPH_SCHEMA_NOTE = """
Node types:
- Restaurant
- Review
- Attribute
- Area
- Cuisine
- Category
- PriceBand
- AtmosphereTag

Relations:
- (Review)-[:ABOUT]->(Restaurant)
- (Restaurant)-[:HAS_ATTRIBUTE]->(Attribute)
- (Restaurant)-[:IN_AREA]->(Area)
- (Restaurant)-[:HAS_CUISINE]->(Cuisine)
- (Restaurant)-[:HAS_CATEGORY]->(Category)
- (Restaurant)-[:HAS_PRICE_BAND]->(PriceBand)
- (Restaurant)-[:HAS_ATMOSPHERE]->(AtmosphereTag)
"""
print(GRAPH_SCHEMA_NOTE)


In [ ]:
from neo4j import GraphDatabase

class Neo4jClient:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
    def close(self):
        self.driver.close()
    def run(self, query: str, params: dict | None = None):
        with self.driver.session() as s:
            return [r.data() for r in s.run(query, params or {})]
    def create_schema(self):
        stmts = [
            "CREATE CONSTRAINT restaurant_key IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.store_key IS UNIQUE",
            "CREATE CONSTRAINT review_key IF NOT EXISTS FOR (r:Review) REQUIRE r.review_id IS UNIQUE",
            "CREATE CONSTRAINT area_name IF NOT EXISTS FOR (a:Area) REQUIRE (a.name, a.city) IS NODE KEY",
            "CREATE CONSTRAINT cuisine_name IF NOT EXISTS FOR (c:Cuisine) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT priceband_name IF NOT EXISTS FOR (p:PriceBand) REQUIRE p.name IS UNIQUE",
            "CREATE CONSTRAINT atmos_name IF NOT EXISTS FOR (a:AtmosphereTag) REQUIRE a.name IS UNIQUE",
            "CREATE INDEX rest_rating IF NOT EXISTS FOR (r:Restaurant) ON (r.gmaps_rating)",
            "CREATE INDEX attr_type IF NOT EXISTS FOR (a:Attribute) ON (a.type)",
        ]
        for q in stmts:
            try:
                self.run(q)
            except Exception as e:
                print("skip schema:", e)

try:
    neo4j_client = Neo4jClient(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    neo4j_client.create_schema()
    print("✅ Neo4j connected")
except Exception as e:
    neo4j_client = None
    print("⚠️ Neo4j unavailable:", e)


In [ ]:
def upsert_restaurants_graph(client: Neo4jClient, restaurants_df: pd.DataFrame):
    rows = []
    for _, r in restaurants_df.iterrows():
        rows.append({
            "store_key": r["store_key"], "name": r["name"], "address": r["address"],
            "district": r["district"], "city": r["city"], "lat": r["lat"], "lng": r["lng"],
            "gmaps_rating": r["gmaps_rating"], "foody_rating": r["foody_rating"], "price_band": r["price_band"],
            "categories": r["categories"], "cuisines": r["cuisines"], "atmosphere": r["atmosphere"], "audiences": r["audiences"],
        })
    q = """
    UNWIND $rows AS row
    MERGE (r:Restaurant {store_key: row.store_key})
    SET r.name = row.name, r.address = row.address, r.district = row.district, r.city = row.city,
        r.lat = row.lat, r.lng = row.lng, r.gmaps_rating = row.gmaps_rating, r.foody_rating = row.foody_rating,
        r.audiences = row.audiences, r.updated_at = datetime()
    WITH r, row
    FOREACH (cat IN coalesce(row.categories, []) | MERGE (c:Category {name: cat}) MERGE (r)-[:HAS_CATEGORY]->(c))
    FOREACH (cui IN coalesce(row.cuisines, []) | MERGE (c:Cuisine {name: cui}) MERGE (r)-[:HAS_CUISINE]->(c))
    FOREACH (atm IN coalesce(row.atmosphere, []) | MERGE (a:AtmosphereTag {name: atm}) MERGE (r)-[:HAS_ATMOSPHERE]->(a))
    FOREACH (_ IN CASE WHEN row.price_band IS NULL THEN [] ELSE [1] END | MERGE (p:PriceBand {name: row.price_band}) MERGE (r)-[:HAS_PRICE_BAND]->(p))
    FOREACH (_ IN CASE WHEN row.district IS NULL OR row.city IS NULL THEN [] ELSE [1] END | MERGE (a:Area {name: row.district, city: row.city}) MERGE (r)-[:IN_AREA]->(a))
    """
    client.run(q, {"rows": rows})

def upsert_attributes_graph(client: Neo4jClient, attrs_df: pd.DataFrame):
    rows = attrs_df.to_dict(orient="records")
    q = """
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (a:Attribute {store_key: row.store_key, type: row.attribute_type})
    SET a.score = row.attribute_score, a.sample_count = row.sample_count
    MERGE (r)-[:HAS_ATTRIBUTE]->(a)
    """
    client.run(q, {"rows": rows})

def upsert_reviews_graph(client: Neo4jClient, reviews_df: pd.DataFrame):
    rows = []
    for _, r in reviews_df.iterrows():
        rows.append({
            "review_id": r["review_id"], "store_key": r["store_key"], "feedback": r["feedback"],
            "rating": r["rating"], "rated_at": str(r["rated_at"]), "sentiment": r["sentiment"],
            "aspect_scores": json.dumps(r["aspect_scores"], ensure_ascii=False), "source": r["source"],
        })
    q = """
    UNWIND $rows AS row
    MATCH (rest:Restaurant {store_key: row.store_key})
    MERGE (rv:Review {review_id: row.review_id})
    SET rv.feedback = row.feedback, rv.rating = row.rating, rv.rated_at = row.rated_at,
        rv.sentiment = row.sentiment, rv.aspect_scores = row.aspect_scores, rv.source = row.source
    MERGE (rv)-[:ABOUT]->(rest)
    """
    client.run(q, {"rows": rows})

if neo4j_client:
    upsert_restaurants_graph(neo4j_client, summary)
    upsert_attributes_graph(neo4j_client, restaurant_attrs)
    upsert_reviews_graph(neo4j_client, feedback_proc)
    print("✅ Graph upsert complete")
else:
    print("⚠️ Skip graph upsert because Neo4j is unavailable")


## 6. Similarity edges + neighborhood expansion

In [ ]:
def build_similarity_edges(client: Neo4jClient, attrs_df: pd.DataFrame, threshold: float = 0.45):
    aspects = sorted(attrs_df["attribute_type"].unique().tolist())
    store_to_vec = defaultdict(lambda: np.zeros(len(aspects), dtype=float))
    idx = {a:i for i,a in enumerate(aspects)}
    for _, row in attrs_df.iterrows():
        store_to_vec[row["store_key"]][idx[row["attribute_type"]]] = row["attribute_score"]
    store_keys = list(store_to_vec.keys())
    mat = np.vstack([store_to_vec[s] for s in store_keys])
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9
    sim = (mat / norms) @ (mat / norms).T
    edges = []
    for i in range(len(store_keys)):
        for j in range(i+1, len(store_keys)):
            if sim[i, j] >= threshold:
                edges.append({"a": store_keys[i], "b": store_keys[j], "sim": round(float(sim[i, j]), 4)})
    if edges:
        client.run("""
        UNWIND $edges AS e
        MATCH (a:Restaurant {store_key: e.a})
        MATCH (b:Restaurant {store_key: e.b})
        MERGE (a)-[s:SIMILAR_TO]-(b)
        SET s.similarity = e.sim
        """, {"edges": edges})
    return len(edges)

if neo4j_client:
    print("✅ Similarity edges:", build_similarity_edges(neo4j_client, restaurant_attrs, threshold=0.45))

def get_restaurant_subgraph(store_key: str, top_reviews: int = 3) -> Dict[str, Any]:
    if neo4j_client is None:
        return {}
    q = """
    MATCH (r:Restaurant {store_key: $store_key})
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(c:Cuisine)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(g:Category)
    OPTIONAL MATCH (r)-[:HAS_ATMOSPHERE]->(t:AtmosphereTag)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)<-[:ABOUT]-(rv:Review)
    WITH r, a, collect(DISTINCT c.name) AS cuisines, collect(DISTINCT g.name) AS categories,
         collect(DISTINCT t.name) AS atmos, collect(DISTINCT {type: att.type, score: att.score}) AS attrs,
         collect(DISTINCT {feedback: rv.feedback, sentiment: rv.sentiment, rating: rv.rating})[..$top_reviews] AS reviews
    OPTIONAL MATCH (r)-[s:SIMILAR_TO]-(nbr:Restaurant)
    WITH r, a, cuisines, categories, atmos, attrs, reviews,
         collect(DISTINCT {store_key: nbr.store_key, name: nbr.name, similarity: s.similarity})[..5] AS neighbors
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address, r.gmaps_rating AS rating,
           a.name AS district, a.city AS city, cuisines, categories, atmos, attrs, reviews, neighbors
    """
    rows = neo4j_client.run(q, {"store_key": store_key, "top_reviews": top_reviews})
    return rows[0] if rows else {}


## 7. Embeddings và Qdrant

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = len(embed_model.encode("passage: test", normalize_embeddings=True))
print("✅ Embedding dim:", EMBED_DIM)

def emb_passage(text: str) -> List[float]:
    return embed_model.encode("passage: " + text, normalize_embeddings=True).tolist()

def emb_query(text: str) -> List[float]:
    return embed_model.encode("query: " + text, normalize_embeddings=True).tolist()

try:
    qdrant = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
    print("✅ Qdrant connected")
except Exception as e:
    qdrant = None
    print("⚠️ Qdrant unavailable:", e)

def ensure_collection(client, name: str, dim: int):
    names = [c.name for c in client.get_collections().collections]
    if name not in names:
        client.create_collection(name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
        print("Created:", name)
    else:
        print("Exists:", name)

if qdrant:
    ensure_collection(qdrant, COLL_RESTAURANT, EMBED_DIM)
    ensure_collection(qdrant, COLL_REVIEW, EMBED_DIM)


In [ ]:
def build_restaurant_summary_doc(store_key: str) -> str:
    row = summary.set_index("store_key").loc[store_key]
    attrs = restaurant_attrs[restaurant_attrs["store_key"].eq(store_key)]
    attr_text = ", ".join(f"{r.attribute_type}={r.attribute_score:+.2f}" for _, r in attrs.sort_values("attribute_type").iterrows())
    pos_reviews = feedback_proc[(feedback_proc["store_key"].eq(store_key)) & (feedback_proc["sentiment"].eq("positive"))]["feedback"].head(3).tolist()
    neg_reviews = feedback_proc[(feedback_proc["store_key"].eq(store_key)) & (feedback_proc["sentiment"].eq("negative"))]["feedback"].head(2).tolist()
    return "\n".join([
        f"Tên quán: {row['name']}",
        f"Địa chỉ: {row['address']}",
        f"Khu vực: {row['district']}, {row['city']}",
        f"Rating Google Maps: {row['gmaps_rating']}",
        f"Foody rating: {row['foody_rating']}",
        f"Price band: {row['price_band']}",
        f"Categories: {', '.join(row['categories'] or [])}",
        f"Cuisines: {', '.join(row['cuisines'] or [])}",
        f"Atmosphere: {', '.join(row['atmosphere'] or [])}",
        f"Audience: {', '.join(row['audiences'] or [])}",
        f"Aggregated aspects: {attr_text}",
        f"Review tích cực mẫu: {' | '.join(pos_reviews)}",
        f"Review chưa tốt mẫu: {' | '.join(neg_reviews)}",
    ])

restaurant_docs = pd.DataFrame({
    "store_key": summary["store_key"],
    "doc_text": summary["store_key"].apply(build_restaurant_summary_doc)
})

def stable_int_id(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

def index_restaurant_docs():
    if qdrant is None:
        return
    points = []
    for _, row in restaurant_docs.iterrows():
        meta = summary.set_index("store_key").loc[row["store_key"]]
        points.append(PointStruct(
            id=stable_int_id("rest-" + row["store_key"]),
            vector=emb_passage(row["doc_text"]),
            payload={
                "store_key": row["store_key"], "name": meta["name"], "address": meta["address"], "district": meta["district"],
                "city": meta["city"], "rating": meta["gmaps_rating"], "price_band": meta["price_band"],
                "doc_text": row["doc_text"], "doc_type": "restaurant_summary",
            }
        ))
    qdrant.upsert(collection_name=COLL_RESTAURANT, points=points)
    print(f"✅ Indexed restaurant docs: {len(points)}")

def index_review_chunks():
    if qdrant is None:
        return
    points = []
    for _, row in review_chunks.iterrows():
        points.append(PointStruct(
            id=stable_int_id("rev-" + row["chunk_id"]),
            vector=emb_passage(row["chunk_text"]),
            payload={
                "chunk_id": row["chunk_id"], "review_id": row["review_id"], "store_key": row["store_key"],
                "store_name": name_map.get(row["store_key"], row["store_name"]), "rating": row["rating"], "sentiment": row["sentiment"],
                "feedback": row["feedback"], "aspect_scores": row["aspect_scores"], "doc_text": row["chunk_text"], "doc_type": "review_chunk",
            }
        ))
    qdrant.upsert(collection_name=COLL_REVIEW, points=points)
    print(f"✅ Indexed review chunks: {len(points)}")

if qdrant:
    index_restaurant_docs()
    index_review_chunks()


## 8. Intent parsing tốt hơn: rule-first, LLM fallback

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

def get_llm():
    if ANTHROPIC_API_KEY:
        return ChatAnthropic(model="claude-sonnet-4-20250514", api_key=ANTHROPIC_API_KEY, temperature=0)
    if OPENAI_API_KEY:
        return ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    return None

llm = get_llm()

KNOWN_CUISINES = sorted({x for xs in summary["cuisines"] for x in (xs or [])})
KNOWN_CATEGORIES = sorted({x for xs in summary["categories"] for x in (xs or [])})
KNOWN_DISTRICTS = sorted({str(x) for x in summary["district"].dropna().unique().tolist() if str(x).strip()})

ASPECT_ALIAS = {
    "ngon": "food_quality", "dở": "food_quality", "chất lượng": "food_quality",
    "phục vụ": "service", "nhân viên": "service", "thái độ": "service",
    "sạch": "cleanliness", "sạch sẽ": "cleanliness",
    "đóng gói": "packaging", "giá": "price", "rẻ": "price", "đắt": "price",
    "không gian": "space", "ấm cúng": "space", "nhanh": "speed", "chậm": "speed",
}

def parse_intent_rule(query: str) -> dict:
    q = normalize_text(query)
    out = {"query_type": "search", "district": None, "cuisines": [], "categories": [],
           "dish_name": None, "min_rating": None, "price_band": None,
           "required_attributes": [], "sentiment_pref": None, "top_k": 5}
    for d in KNOWN_DISTRICTS:
        if normalize_text(d) in q:
            out["district"] = d
            break
    for c in KNOWN_CUISINES:
        if normalize_text(c) in q:
            out["cuisines"].append(c)
    for c in KNOWN_CATEGORIES:
        if normalize_text(c) in q:
            out["categories"].append(c)
    if "giá rẻ" in q or "rẻ" in q or "sinh viên" in q:
        out["price_band"] = "budget"
    elif "cao cấp" in q or "sang" in q:
        out["price_band"] = "premium"
    elif "tầm trung" in q:
        out["price_band"] = "mid"
    for k, v in ASPECT_ALIAS.items():
        if k in q and v not in out["required_attributes"]:
            out["required_attributes"].append(v)
    if "tương tự" in q or "giống" in q:
        out["query_type"] = "similar"
    if "cá nhân" in q or "mình thích" in q:
        out["query_type"] = "personalized"
    return out

INTENT_SYSTEM = """Bạn là bộ phân tích truy vấn tìm quán ăn.
Trả về JSON hợp lệ với các trường:
query_type, district, cuisines, categories, dish_name, min_rating, price_band, required_attributes, sentiment_pref, top_k
Không viết gì ngoài JSON.
"""

intent_prompt = ChatPromptTemplate.from_messages([("system", INTENT_SYSTEM), ("human", "{query}")])

def parse_intent(query: str) -> dict:
    base = parse_intent_rule(query)
    if llm is None:
        return base
    try:
        resp = (intent_prompt | llm).invoke({"query": query}).content
        m = re.search(r"\{.*\}", resp, re.DOTALL)
        extra = json.loads(m.group()) if m else {}
        for k, v in extra.items():
            if v not in [None, [], ""]:
                base[k] = v
    except Exception as e:
        print("LLM parse fallback -> using rules only:", e)
    return base


## 9. Graph retrieval + neighborhood / subgraph retrieval

In [ ]:
def graph_candidate_search(intent: dict, top_k: int = 10) -> List[dict]:
    if neo4j_client is None:
        return []
    match_lines = ["MATCH (r:Restaurant)"]
    where = []
    params = {"top_k": int(top_k)}
    if intent.get("district"):
        match_lines.append("MATCH (r)-[:IN_AREA]->(area:Area)")
        where.append("toLower(area.name) CONTAINS toLower($district)")
        params["district"] = intent["district"]
    if intent.get("price_band"):
        match_lines.append("MATCH (r)-[:HAS_PRICE_BAND]->(pb:PriceBand)")
        where.append("pb.name = $price_band")
        params["price_band"] = intent["price_band"]
    if intent.get("cuisines"):
        match_lines.append("MATCH (r)-[:HAS_CUISINE]->(cui:Cuisine)")
        where.append("cui.name IN $cuisines")
        params["cuisines"] = intent["cuisines"]
    if intent.get("categories"):
        match_lines.append("MATCH (r)-[:HAS_CATEGORY]->(cat:Category)")
        where.append("cat.name IN $categories")
        params["categories"] = intent["categories"]
    for i, attr in enumerate(intent.get("required_attributes", [])):
        alias = f"att{i}"
        match_lines.append(f"MATCH (r)-[:HAS_ATTRIBUTE]->({alias}:Attribute {{type: $attr_{i}}})")
        where.append(f"{alias}.score >= 0")
        params[f"attr_{i}"] = attr
    q = "\n".join(match_lines) + "\n"
    if where:
        q += "WHERE " + " AND ".join(where) + "\n"
    q += """
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    WITH r, a, collect(DISTINCT {type: att.type, score: att.score}) AS attributes
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address,
           a.name AS district, a.city AS city, r.gmaps_rating AS rating, attributes
    ORDER BY r.gmaps_rating DESC NULLS LAST
    LIMIT $top_k
    """
    return neo4j_client.run(q, params)

def subgraph_expand_candidates(seed_store_keys: List[str], max_neighbors: int = 3) -> List[dict]:
    if neo4j_client is None or not seed_store_keys:
        return []
    q = """
    UNWIND $keys AS key
    MATCH (r:Restaurant {store_key: key})-[s:SIMILAR_TO]-(nbr:Restaurant)
    OPTIONAL MATCH (nbr)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (nbr)-[:IN_AREA]->(a:Area)
    WITH nbr, max(s.similarity) AS sim, a, collect(DISTINCT {type: att.type, score: att.score}) AS attributes
    RETURN nbr.store_key AS store_key, nbr.name AS name, nbr.address AS address,
           a.name AS district, a.city AS city, nbr.gmaps_rating AS rating, sim, attributes
    ORDER BY sim DESC, rating DESC
    LIMIT $max_neighbors
    """
    return neo4j_client.run(q, {"keys": seed_store_keys, "max_neighbors": max_neighbors})


## 10. Vector retrieval cho summary và review evidence

In [ ]:
def vector_search_restaurants(query: str, top_k: int = 5) -> List[dict]:
    if qdrant is None:
        return []
    hits = qdrant.search(collection_name=COLL_RESTAURANT, query_vector=emb_query(query), limit=top_k, with_payload=True)
    return [{
        "store_key": h.payload["store_key"], "name": h.payload["name"], "address": h.payload.get("address"),
        "district": h.payload.get("district"), "city": h.payload.get("city"), "rating": h.payload.get("rating"),
        "price_band": h.payload.get("price_band"), "doc_text": h.payload.get("doc_text"),
        "vec_score": round(float(h.score), 4), "source": "restaurant_vector",
    } for h in hits]

def vector_search_reviews(query: str, top_k: int = 8, store_keys: Optional[List[str]] = None) -> List[dict]:
    if qdrant is None:
        return []
    hits = qdrant.search(collection_name=COLL_REVIEW, query_vector=emb_query(query), limit=top_k, with_payload=True)
    rows = []
    allow = set(store_keys) if store_keys else None
    for h in hits:
        p = h.payload
        if allow and p.get("store_key") not in allow:
            continue
        rows.append({
            "store_key": p["store_key"], "review_id": p["review_id"], "store_name": p.get("store_name"),
            "rating": p.get("rating"), "sentiment": p.get("sentiment"), "feedback": p.get("feedback"),
            "aspect_scores": p.get("aspect_scores"), "doc_text": p.get("doc_text"),
            "vec_score": round(float(h.score), 4), "source": "review_vector",
        })
    return rows


## 11. Fusion + rerank

In [ ]:
def aggregate_review_evidence(review_hits: List[dict]) -> Dict[str, dict]:
    bucket = defaultdict(lambda: {"evidence": [], "review_vec_score_max": 0.0})
    for r in review_hits:
        b = bucket[r["store_key"]]
        b["evidence"].append(r["feedback"])
        b["review_vec_score_max"] = max(b["review_vec_score_max"], r.get("vec_score", 0.0))
    return bucket

def rerank_candidates(query: str, intent: dict, graph_hits: List[dict], neighbor_hits: List[dict], rest_vec_hits: List[dict], review_vec_hits: List[dict]) -> List[dict]:
    by_store = {}
    def ensure(rec: dict):
        sid = rec["store_key"]
        if sid not in by_store:
            by_store[sid] = {
                "store_key": sid, "name": rec.get("name"), "address": rec.get("address"),
                "district": rec.get("district"), "city": rec.get("city"), "rating": rec.get("rating"),
                "graph_score": 0.0, "neighbor_score": 0.0, "restaurant_vec_score": 0.0, "review_vec_score": 0.0,
                "attribute_match_score": 0.0, "evidence": [], "source_flags": set(),
            }
        return by_store[sid]
    for r in graph_hits:
        c = ensure(r); c["graph_score"] = max(c["graph_score"], 1.0); c["source_flags"].add("graph")
    for r in neighbor_hits:
        c = ensure(r); c["neighbor_score"] = max(c["neighbor_score"], float(r.get("sim", 0.0))); c["source_flags"].add("neighbor")
    for r in rest_vec_hits:
        c = ensure(r); c["restaurant_vec_score"] = max(c["restaurant_vec_score"], float(r.get("vec_score", 0.0))); c["source_flags"].add("restaurant_vector")
    review_bucket = aggregate_review_evidence(review_vec_hits)
    for sid, info in review_bucket.items():
        c = ensure({"store_key": sid, "name": name_map.get(sid)})
        c["review_vec_score"] = max(c["review_vec_score"], info["review_vec_score_max"])
        c["evidence"] = info["evidence"][:3]
        c["source_flags"].add("review_vector")
    attr_map = restaurant_attrs.groupby("store_key").apply(lambda g: {r["attribute_type"]: r["attribute_score"] for _, r in g.iterrows()}).to_dict()
    for sid, c in by_store.items():
        attrs = attr_map.get(sid, {})
        req = intent.get("required_attributes", []) or []
        if req:
            vals = [max(0.0, (attrs.get(a, -1.0) + 1) / 2) for a in req]
            c["attribute_match_score"] = float(np.mean(vals)) if vals else 0.0
        else:
            c["attribute_match_score"] = 0.5
    ratings = [x.get("rating") for x in by_store.values() if x.get("rating") is not None]
    rmin, rmax = (min(ratings), max(ratings)) if ratings else (0.0, 5.0)
    results = []
    for sid, c in by_store.items():
        rating = c["rating"] if c["rating"] is not None else 0.0
        rating_norm = (rating - rmin) / (rmax - rmin + 1e-9)
        c["final_score"] = (
            0.22 * c["graph_score"] + 0.12 * c["neighbor_score"] + 0.26 * c["restaurant_vec_score"] +
            0.20 * c["review_vec_score"] + 0.10 * c["attribute_match_score"] + 0.10 * rating_norm
        )
        c["source_flags"] = sorted(list(c["source_flags"]))
        results.append(c)
    results.sort(key=lambda x: x["final_score"], reverse=True)
    return results

def hybrid_retrieve(query: str, top_k: int = 5) -> Tuple[dict, List[dict]]:
    intent = parse_intent(query)
    graph_hits = graph_candidate_search(intent, top_k=max(8, top_k))
    seed_keys = [r["store_key"] for r in graph_hits[:3]]
    neighbor_hits = subgraph_expand_candidates(seed_keys, max_neighbors=6) if seed_keys else []
    rest_vec_hits = vector_search_restaurants(query, top_k=max(8, top_k))
    store_scope = list({*(r["store_key"] for r in graph_hits), *(r["store_key"] for r in rest_vec_hits)})
    review_vec_hits = vector_search_reviews(query, top_k=12, store_keys=store_scope if store_scope else None)
    ranked = rerank_candidates(query, intent, graph_hits, neighbor_hits, rest_vec_hits, review_vec_hits)
    return intent, ranked[:top_k]


## 12. Generation / recommendation

In [ ]:
RECOMMEND_SYSTEM = """Bạn là trợ lý gợi ý quán ăn.
Dựa trên kết quả đã được truy hồi và rerank, hãy trả lời ngắn gọn bằng tiếng Việt:
- nêu top quán phù hợp nhất
- giải thích vì sao phù hợp
- trích dẫn evidence review nếu có
Không bịa thông tin ngoài context.
"""

recommend_prompt = ChatPromptTemplate.from_messages([("system", RECOMMEND_SYSTEM), ("human", "Query: {query}\n\nContext:\n{context}")])

def format_retrieval_context(rows: List[dict]) -> str:
    lines = []
    for i, r in enumerate(rows, 1):
        ev = " | ".join(r.get("evidence", [])[:2]) if r.get("evidence") else ""
        lines.append(
            f"{i}. {r.get('name')} | rating={r.get('rating')} | district={r.get('district')} | "
            f"score={r.get('final_score'):.3f} | sources={','.join(r.get('source_flags', []))}\n"
            f"   address={r.get('address')}\n"
            f"   evidence={ev}"
        )
    return "\n\n".join(lines)

def recommend(query: str, top_k: int = 5) -> str:
    intent, ranked = hybrid_retrieve(query, top_k=top_k)
    context = format_retrieval_context(ranked)
    if llm is None:
        return context
    return (recommend_prompt | llm).invoke({"query": query, "context": context}).content


## 13. Evaluation cho retrieval quality

In [ ]:
def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant); ret = retrieved[:k]
    return len(rel.intersection(ret)) / len(rel) if rel else 0.0

def mrr_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    for i, sid in enumerate(retrieved[:k], start=1):
        if sid in rel:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    dcg = 0.0
    for i, sid in enumerate(retrieved[:k], start=1):
        gain = 1.0 if sid in rel else 0.0
        dcg += gain / math.log2(i + 1)
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_retrieval(test_cases: List[dict], k: int = 5) -> pd.DataFrame:
    rows = []
    for tc in test_cases:
        _, ranked = hybrid_retrieve(tc["query"], top_k=k)
        retrieved = [r["store_key"] for r in ranked]
        rows.append({
            "query": tc["query"], "relevant": tc["relevant_store_keys"],
            f"recall@{k}": recall_at_k(retrieved, tc["relevant_store_keys"], k),
            f"mrr@{k}": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
            f"ndcg@{k}": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            "retrieved": retrieved,
        })
    return pd.DataFrame(rows)

demo_test_cases = [
    {"query": "mì cay hàn quốc", "relevant_store_keys": ["32441"]},
    {"query": "quán sạch sẽ phục vụ tốt ở Hai Bà Trưng", "relevant_store_keys": ["32441"]},
    {"query": "quán giá rẻ cho sinh viên", "relevant_store_keys": summary["store_key"].head(3).tolist()},
]


## 14. Ablation gợi ý

In [ ]:
def retrieve_vector_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_restaurants(query, top_k=top_k)]

def retrieve_graph_only(query: str, top_k: int = 5) -> List[str]:
    intent = parse_intent(query)
    return [r["store_key"] for r in graph_candidate_search(intent, top_k=top_k)]

def retrieve_hybrid(query: str, top_k: int = 5) -> List[str]:
    _, ranked = hybrid_retrieve(query, top_k=top_k)
    return [r["store_key"] for r in ranked]


## 15. Kết luận method

### Những gì đã cải thiện so với bản trước
- graph **không còn nông**: đã thêm node riêng cho `Cuisine`, `Category`, `PriceBand`, `AtmosphereTag`
- review evidence **đã chunk hóa riêng**
- đã có **neighborhood / subgraph retrieval**
- đã có **evaluation retrieval quality**
- `parse_intent` chuyển sang **rule-first + LLM fallback**
- embedding được tách rõ thành:
  - **restaurant summary embeddings**
  - **review chunk embeddings**

### Tên method phù hợp
**Hybrid Graph-enhanced RAG for restaurant recommendation**
